In [1]:
import pandas as pd
df = pd.read_csv("brain_stroke.csv")
print("Success! Data shape:", df.shape)
df.head()

Success! Data shape: (4981, 11)


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

X = df.drop(columns=['stroke'])
y = df['stroke']

categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
numerical_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}
print("Preprocessed Input Feature Shape:", X_train.shape)

Preprocessed Input Feature Shape: (3984, 19)


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Add

# 1. Ensure absolute reproducibility
tf.random.set_seed(42)

# =====================================================================
# 🏗️ Model 1: Vanilla Tabular MLP (The Baseline)
# =====================================================================
model_1 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
history_1 = model_1.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, verbose=0)
print("✔ Model 1 (Vanilla) trained completely.")

# =====================================================================
# 🛡️ Model 2: Regularized MLP (With Dropout & Batch Normalization)
# =====================================================================
model_2 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
history_2 = model_2.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, verbose=0)
print("✔ Model 2 (Regularized) trained completely.")

# =====================================================================
# 🎯 Model 3: Class-Weighted MLP (Handling 5% minority imbalance)
# =====================================================================
model_3 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
history_3 = model_3.fit(X_train, y_train, epochs=30, batch_size=32,
                        class_weight=class_weight_dict, validation_split=0.1, verbose=0)
print("✔ Model 3 (Class-Weighted) trained completely.")

# =====================================================================
# ⚡ Model 4: Residual Tabular MLP (With Functional Skip Connections)
# =====================================================================
inputs = Input(shape=(X_train.shape[1],))
x1 = Dense(64, activation='relu')(inputs)
x2 = Dense(64, activation='relu')(x1)
skip_node = Add()([x1, x2])  # Residual shortcut
x3 = Dense(32, activation='relu')(skip_node)
outputs = Dense(1, activation='sigmoid')(x3)

model_4 = Model(inputs=inputs, outputs=outputs)
model_4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
history_4 = model_4.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, verbose=0)
print("✔ Model 4 (Residual Structure) trained completely.")

✔ Model 1 (Vanilla) trained completely.
✔ Model 2 (Regularized) trained completely.
✔ Model 3 (Class-Weighted) trained completely.
✔ Model 4 (Residual Structure) trained completely.


In [4]:
from sklearn.metrics import f1_score, precision_recall_curve, auc

models = [model_1, model_2, model_3, model_4]
model_names = ["Vanilla MLP", "Regularized MLP", "Class-Weighted MLP", "Residual MLP"]

print("\n--- 📊 GSSoC PROJECT EVALUATION MATRIX ---")
for name, model in zip(model_names, models):
    preds = model.predict(X_test, verbose=0)
    preds_binary = (preds > 0.5).astype(int)

    f1 = f1_score(y_test, preds_binary)
    precision, recall, _ = precision_recall_curve(y_test, preds)
    pr_auc = auc(recall, precision)

    print(f"{name:<20} -> F1-Score: {f1:.4f} | PR-AUC: {pr_auc:.4f}")


--- 📊 GSSoC PROJECT EVALUATION MATRIX ---
Vanilla MLP          -> F1-Score: 0.0377 | PR-AUC: 0.1938
Regularized MLP      -> F1-Score: 0.0392 | PR-AUC: 0.1729
Class-Weighted MLP   -> F1-Score: 0.2182 | PR-AUC: 0.1760
Residual MLP         -> F1-Score: 0.1127 | PR-AUC: 0.1603
